# 5.18 Hierarchical Bayesian Models and Practical Interpretation

[Back to statistics index](../5_statics.ipynb)

This notebook collects all six subparagraphs for **Hierarchical Bayesian Models and Practical Interpretation**. Each subsection keeps its explanation close to its own worked example, and imports only what it needs.


## 5.18.1 Why pooling matters

### Why It Matters
Separate small-group estimates are noisy, while one global estimate can hide meaningful subgroup differences.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(300)
groups = {
    "north": rng.normal(78, 5, 60),
    "south": rng.normal(74, 5, 18),
    "east": rng.normal(76, 5, 10),
}
pooled_mean = np.concatenate(list(groups.values())).mean()
separate = {k: v.mean() for k, v in groups.items()}
pd.DataFrame({"group_mean": separate, "distance_from_pooled": {k: separate[k] - pooled_mean for k in separate}}).round(2)


### Interpretation
- This notebook isolates **why pooling matters** instead of combining it with the rest of the chapter.
- The code cell is meant to be edited and rerun as part of the lecture.
- When you are comfortable with this subsection, return to the index and continue to the next notebook.


## 5.18.2 Partial pooling intuition

### Why It Matters
Hierarchical models shrink unstable group estimates toward a common center, with more shrinkage for data-poor groups.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(301)
group_means = pd.Series({
    "large_group": rng.normal(80, 1, 200).mean(),
    "small_group": rng.normal(86, 1, 12).mean(),
})
group_sizes = pd.Series({"large_group": 200, "small_group": 12})
global_mean = np.average(group_means, weights=group_sizes)
shrink_weight = group_sizes / (group_sizes + 40)
partially_pooled = shrink_weight * group_means + (1 - shrink_weight) * global_mean
pd.DataFrame({"raw_group_mean": group_means, "partially_pooled_mean": partially_pooled, "group_size": group_sizes}).round(2)


### Interpretation
- This notebook isolates **partial pooling intuition** instead of combining it with the rest of the chapter.
- The code cell is meant to be edited and rerun as part of the lecture.
- When you are comfortable with this subsection, return to the index and continue to the next notebook.


## 5.18.3 Multilevel structure

### Why It Matters
A group-level distribution governs the group-specific parameters, which then govern the observations.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(302)
df = pd.DataFrame({
    "region": np.repeat(["north", "south", "east"], [4, 3, 2]),
    "month": [1, 2, 3, 4, 1, 2, 3, 1, 2],
    "sales_lift": rng.normal(5, 1, 9),
})
pd.DataFrame({
    "global_mean": [df["sales_lift"].mean()],
    "group_means": [df.groupby("region")["sales_lift"].mean().round(2).to_dict()],
    "observation_count": [len(df)],
})


### Interpretation
- This notebook isolates **multilevel structure** instead of combining it with the rest of the chapter.
- The code cell is meant to be edited and rerun as part of the lecture.
- When you are comfortable with this subsection, return to the index and continue to the next notebook.


## 5.18.4 When hierarchical models help

### Why It Matters
Regional performance, classrooms, experiments by channel, and repeated studies are classic use cases.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(303)
group_data = {
    "large_region": rng.normal(82, 4, 120),
    "small_region": rng.normal(90, 4, 8),
    "medium_region": rng.normal(85, 4, 35),
}
summary = pd.DataFrame({
    "group_mean": {k: v.mean() for k, v in group_data.items()},
    "group_size": {k: len(v) for k, v in group_data.items()},
})
global_mean = np.average(summary["group_mean"], weights=summary["group_size"])
summary["distance_from_global"] = summary["group_mean"] - global_mean
summary.round(2)


### Interpretation
- This notebook isolates **when hierarchical models help** instead of combining it with the rest of the chapter.
- The code cell is meant to be edited and rerun as part of the lecture.
- When you are comfortable with this subsection, return to the index and continue to the next notebook.


## 5.18.5 Interpretation of shrinkage

### Why It Matters
Shrinkage is not suppression. It is the model expressing justified caution about small-sample groups.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(304)
group_data = {
    "north": rng.normal(82, 4, 100),
    "south": rng.normal(89, 4, 6),
    "east": rng.normal(80, 4, 14),
}
raw_means = pd.Series({k: v.mean() for k, v in group_data.items()})
sizes = pd.Series({k: len(v) for k, v in group_data.items()})
global_mean = np.average(raw_means, weights=sizes)
weights = sizes / (sizes + 35)
shrunk_means = weights * raw_means + (1 - weights) * global_mean
pd.DataFrame({"raw_mean": raw_means, "shrunk_mean": shrunk_means, "group_size": sizes}).round(2)


### Interpretation
- This notebook isolates **interpretation of shrinkage** instead of combining it with the rest of the chapter.
- The code cell is meant to be edited and rerun as part of the lecture.
- When you are comfortable with this subsection, return to the index and continue to the next notebook.


## 5.18.6 Computation and realism

### Why It Matters
Hierarchical models are richer and often more realistic, but they require stronger diagnostic discipline.


In [ ]:
import arviz as az
import numpy as np
import pymc as pm

rng = np.random.default_rng(305)
region = np.repeat([0, 1, 2], [24, 14, 10])
ad_spend = rng.normal(10, 2, len(region))
offsets = np.array([1.5, -0.5, 0.8])
sales_lift = offsets[region] + 0.9 * ad_spend + rng.normal(0, 1.1, len(region))

with pm.Model(coords={"region": ["north", "south", "east"]}) as model:
    mu_a = pm.Normal("mu_a", 0, 3)
    sigma_a = pm.Exponential("sigma_a", 1)
    a = pm.Normal("a", mu=mu_a, sigma=sigma_a, dims="region")
    beta = pm.Normal("beta", 0, 2)
    sigma = pm.Exponential("sigma", 1)
    mu = a[region] + beta * ad_spend
    pm.Normal("sales_lift", mu=mu, sigma=sigma, observed=sales_lift)
    trace = pm.sample(draws=200, tune=200, chains=2, cores=1, progressbar=False, random_seed=42)

az.summary(trace, var_names=["mu_a", "sigma_a", "beta", "sigma"])


### Interpretation
- This notebook isolates **computation and realism** instead of combining it with the rest of the chapter.
- The code cell is meant to be edited and rerun as part of the lecture.
- When you are comfortable with this subsection, return to the index and continue to the next notebook.
